# Chronumental dating summary

Validates chronumental's predicted dates by comparing predicted leaf dates against reported `collection_date` from the augmented metadata, across every (segment, subtype) final tree. Writes a self-contained HTML report to `snakemake.output.html`.

In [ ]:
import base64
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

inferred_paths = sorted(snakemake.input.inferred)
metadata_path = snakemake.input.metadata
output_html = snakemake.output.html


def parse_segment_subtype(path):
    """results/HA/H1/chronumental/strain_date_inferred.tsv -> ('HA', 'H1')."""
    parts = Path(path).parts
    return parts[-4], parts[-3]

In [ ]:
metadata = pd.read_csv(metadata_path, usecols=["isolate_id", "collection_date"])
metadata["reported_date"] = pd.to_datetime(metadata["collection_date"], format="%Y-%m-%d", errors="coerce")
metadata = metadata.dropna(subset=["reported_date"])[["isolate_id", "reported_date"]]
print(f"Metadata rows with full YYYY-MM-DD dates: {len(metadata)}")

In [ ]:
rows = []
tree_data = {}
for path in inferred_paths:
    segment, subtype = parse_segment_subtype(path)
    inferred = pd.read_csv(path, sep="\t", usecols=["strain", "predicted_date"])
    inferred["predicted_date"] = pd.to_datetime(inferred["predicted_date"])

    joined = inferred.merge(metadata, left_on="strain", right_on="isolate_id", how="inner")
    joined["residual_days"] = (joined["predicted_date"] - joined["reported_date"]).dt.total_seconds() / 86400.0
    joined["abs_residual_days"] = joined["residual_days"].abs()

    n_total_nodes = len(inferred)
    n_validated_leaves = len(joined)
    n_unvalidated = n_total_nodes - n_validated_leaves

    rows.append({
        "segment": segment,
        "subtype": subtype,
        "n_total_nodes": n_total_nodes,
        "n_validated_leaves": n_validated_leaves,
        "n_internal_or_unmatched": n_unvalidated,
        "median_abs_residual_days": joined["abs_residual_days"].median() if n_validated_leaves else np.nan,
        "mean_abs_residual_days": joined["abs_residual_days"].mean() if n_validated_leaves else np.nan,
        "frac_within_30_days": (joined["abs_residual_days"] <= 30).mean() if n_validated_leaves else np.nan,
        "frac_within_180_days": (joined["abs_residual_days"] <= 180).mean() if n_validated_leaves else np.nan,
        "inferred_min_date": inferred["predicted_date"].min().date().isoformat(),
        "inferred_max_date": inferred["predicted_date"].max().date().isoformat(),
    })
    tree_data[(segment, subtype)] = joined

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
keys = sorted(tree_data.keys())
labels = [f"{seg}/{sub}" for seg, sub in keys]

fig_box, ax = plt.subplots(figsize=(max(8, 0.6 * len(keys)), 5))
ax.boxplot(
    [tree_data[k]["abs_residual_days"].values for k in keys],
    labels=labels,
    showfliers=False,
)
ax.set_ylabel("|predicted - reported| (days)")
ax.set_title("Leaf-date residuals per tree (outliers hidden)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
ncols = 4
nrows = (len(keys) + ncols - 1) // ncols
fig_scatter, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows), squeeze=False)
axes_flat = axes.flatten()
for ax, key in zip(axes_flat, keys):
    j = tree_data[key]
    ax.scatter(j["reported_date"], j["predicted_date"], s=2, alpha=0.3)
    if len(j):
        lo = min(j["reported_date"].min(), j["predicted_date"].min())
        hi = max(j["reported_date"].max(), j["predicted_date"].max())
        ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(f"{key[0]}/{key[1]}")
    ax.set_xlabel("reported")
    ax.set_ylabel("predicted")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for ax in axes_flat[len(keys):]:
    ax.set_visible(False)
fig_scatter.suptitle("Predicted vs reported leaf collection dates")
plt.tight_layout()
plt.show()

In [ ]:
def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    return base64.b64encode(buf.getvalue()).decode("ascii")

html = [
    "<!DOCTYPE html>",
    "<html><head><meta charset='utf-8'><title>Chronumental dating summary</title>",
    "<style>body{font-family:sans-serif;max-width:1200px;margin:20px auto;padding:0 20px;}",
    "table{border-collapse:collapse;font-size:13px;}td,th{border:1px solid #ccc;padding:4px 8px;text-align:right;}th{background:#eee;}",
    "img{max-width:100%;height:auto;}h2{margin-top:30px;}</style></head><body>",
    "<h1>Chronumental dating summary</h1>",
    "<p>Per-tree validation of chronumental's leaf-date predictions against reported <code>collection_date</code> from the augmented metadata.</p>",
    "<h2>Per-tree summary</h2>",
    summary_df.round(2).to_html(index=False),
    "<h2>Leaf-date residuals per tree</h2>",
    f"<img src='data:image/png;base64,{fig_to_b64(fig_box)}'/>",
    "<h2>Predicted vs reported leaf collection dates</h2>",
    f"<img src='data:image/png;base64,{fig_to_b64(fig_scatter)}'/>",
    "</body></html>",
]
Path(output_html).write_text("\n".join(html))
print(f"Wrote {output_html}")